# CopyLane 판정 인코더 2차 골든셋 재학습

**목적:** 2026-09-25 판 골든셋으로 `beomi/kcbert-base`를 새로 학습한다. 이 실험의 출력은 09-22의 8유형 모델과 다른 **조문 호 분류 모델**이다. 점수를 직접 비교하지 않는다.

**입력:** 같은 판의 `golden.jsonl`(8,432행)과 `split_manifest.json`. `injected_golden.jsonl`은 올리거나 다시 합치지 않는다. 골든셋 안에 주입 행 784개가 이미 있다.

**분할:** 기존 `train`만 학습에 쓴다. `train` 안에서 문서 단위 내부 검증을 만든다. `test_sentence`는 학습과 모델 선택에 쓰지 않고 마지막에 한 번만 평가한다.

**평가:** 정답은 `근거`의 조문 호다(D-282). 조건 `D`·`M`은 채점에서 제외한다(D-286). `근거_후보`는 후보 중 예측과 가장 많이 겹치는 것을 정답으로 택한다. 30건 미만 호는 측정 불가로 표시한다.

이 노트북은 원문 데이터나 모델을 자동으로 외부에 업로드하지 않는다. 실행 환경에서 데이터 파일 경로를 지정한다. 데이터에 `redistributable=false` 행이 있으므로 팀의 승인된 저장 경로를 사용한다.

## 1 환경과 파일 경로

패키지가 없는 환경이라면 다음 셀 맨 앞의 `#`를 지운 뒤 실행한다. 이미 설치돼 있다면 건너뛴다.

In [ ]:
# %pip install -q "transformers>=4.46,<6" "datasets>=3,<5" "accelerate>=1,<2" "scikit-learn>=1.5,<2" safetensors


In [ ]:
from pathlib import Path
import hashlib, json, re
import numpy as np

GOLDEN = Path('data/derived/golden/golden.jsonl')
MANIFEST = Path('data/derived/golden/split_manifest.json')
OUTPUT = Path('copylane-encoder-ho-20260925-gamma1.5-cap5')
SEED = 42
GAMMA = 1.5
POS_WEIGHT_CAP = 5.0
THRESHOLD = 0.5
EXPECTED = {'total': 8432, 'train': 6425, 'test_sentence': 2007}

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

for path in (GOLDEN, MANIFEST):
    if not path.is_file():
        raise FileNotFoundError(f'{path} 없음 — 같은 판의 2차 데이터를 먼저 준비하세요')
print('golden sha256', sha256(GOLDEN))
print('manifest sha256', sha256(MANIFEST))


## 2 데이터 계약 확인

행 수, 조문 인용, 문서별 분할을 먼저 검사한다. 09-22 골든셋이 남아 있거나 두 파일의 판이 다르면 학습 전에 멈춘다.

In [ ]:
with GOLDEN.open(encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
assign = manifest['assign']

if len(rows) != EXPECTED['total']:
    raise ValueError(f'골든셋 {len(rows)}행: 기대 {EXPECTED["total"]}행. 새 판을 준비하세요')
if len({r['id'] for r in rows}) != len(rows):
    raise ValueError('중복 id가 있습니다')

CITE = re.compile(r'^\d+:제\d+조제\d+항제\d+호(?:\|[가-힣]목)?$')
def ho_key(citation):
    if not CITE.fullmatch(citation):
        raise ValueError(f'알 수 없는 근거 인용: {citation!r}')
    return citation.split('|', 1)[0]

def source_doc_id(r):
    rid = r['id']
    if rid.startswith('inj:'):
        parts = rid.split(':', 2)
        if len(parts) != 3:
            raise ValueError(f'주입 id에 원본 문서가 없음: {rid}')
        return parts[2]
    if '#' not in rid:
        raise ValueError(f'문서 id를 읽을 수 없음: {rid}')
    return rid.rsplit('#', 1)[0]

from collections import Counter
split_counts = Counter()
condition_counts = Counter()
for r in rows:
    if not {'id','text','근거','labels','unit','split'} <= r.keys():
        raise ValueError(f'필수 필드 누락: {r.get("id")}')
    split = r['split']
    if split not in ('train', 'test_sentence'):
        raise ValueError(f'예상하지 않은 split {split!r}: {r["id"]}')
    source = source_doc_id(r)
    if assign.get(source) != split:
        raise ValueError(f'분할 원장 불일치: {r["id"]} / {assign.get(source)} / {split}')
    for c in r['근거']:
        ho_key(c)
    for candidate in r.get('근거_후보') or []:
        for c in candidate:
            ho_key(c)
    if split == 'train' and r.get('조건') in ('D','M'):
        raise ValueError(f'채점 제외 조건이 학습에 있음: {r["id"]}')
    split_counts[split] += 1
    if r.get('조건'):
        condition_counts[r['조건']] += 1

if split_counts != {'train': EXPECTED['train'], 'test_sentence': EXPECTED['test_sentence']}:
    raise ValueError(f'분할 행 수 불일치: {split_counts}')
if any(r['id'].startswith('inj:') and r['split'] != 'train' for r in rows):
    raise ValueError('주입 행이 평가에 들어갔습니다')
print('분할', dict(split_counts), '조건', dict(condition_counts))
print('학습 단위', dict(Counter(r['unit'] for r in rows if r['split']=='train')))


## 3 학습용 호 라벨과 내부 검증

유형 이름을 분류 라벨로 쓰지 않는다. 같은 호의 목은 하나의 호로 접는다. 봉인 평가에만 나타나는 호도 출력 차원에 남겨 두고, 학습 표본 0개라고 별도 표시한다.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

train_rows = [r for r in rows if r['split'] == 'train']
test_rows = [r for r in rows if r['split'] == 'test_sentence']
all_ho = sorted({ho_key(c) for r in rows for c in r['근거']}
                | {ho_key(c) for r in rows for cand in (r.get('근거_후보') or []) for c in cand})
if not all_ho:
    raise ValueError('호 라벨이 없습니다')
label2id = {label: i for i, label in enumerate(all_ho)}
id2label = {i: label for label, i in label2id.items()}

def vector(r):
    v = np.zeros(len(all_ho), dtype=np.float32)
    for c in r['근거']:
        v[label2id[ho_key(c)]] = 1
    return v

groups = [source_doc_id(r) for r in train_rows]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
fit_idx, dev_idx = next(splitter.split(train_rows, groups=groups))
fit_rows = [train_rows[i] for i in fit_idx]
dev_rows = [train_rows[i] for i in dev_idx]
assert {source_doc_id(r) for r in fit_rows}.isdisjoint({source_doc_id(r) for r in dev_rows})
assert {source_doc_id(r) for r in train_rows}.isdisjoint({source_doc_id(r) for r in test_rows})

fit_counts = np.stack([vector(r) for r in fit_rows]).sum(axis=0).astype(int)
print('train 내부 학습', len(fit_rows), '내부 검증', len(dev_rows), '봉인 평가', len(test_rows))
for label, count in zip(all_ho, fit_counts):
    print(f'{label}: 내부 학습 양성 {count}' + ('  [학습 불가]' if count == 0 else ''))


## 4 kcbert-base 학습

09-22 채택값 γ=1.5, CAP=5를 **새 데이터 실험의 시작 설정**으로 쓴다. 모델 선택은 내부 검증의 micro F1로만 한다. 봉인 평가셋은 Trainer에 전달하지 않는다.

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from sklearn.metrics import precision_recall_fscore_support

set_seed(SEED)
MODEL_NAME = 'beomi/kcbert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(selected):
    ds = Dataset.from_dict({'text': [r['text'] for r in selected],
                            'labels': [vector(r).tolist() for r in selected]})
    ds = ds.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=128,
                                        padding='max_length'), batched=True)
    ds.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
    return ds

fit_ds, dev_ds = make_dataset(fit_rows), make_dataset(dev_rows)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(all_ho), problem_type='multi_label_classification',
    label2id=label2id, id2label=id2label)
raw_weight = (len(fit_rows) - fit_counts) / np.maximum(fit_counts, 1)
pos_weight = torch.tensor(np.clip(raw_weight, 1, POS_WEIGHT_CAP), dtype=torch.float32)

def metrics(eval_pred):
    logits, truth = eval_pred
    pred = (1 / (1 + np.exp(-logits)) > THRESHOLD).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, pred, average='micro', zero_division=0)
    return {'micro_f1': f1, 'micro_precision': precision, 'micro_recall': recall}

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        output = model(**inputs)
        logits = output.logits
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction='none')
        prob = torch.sigmoid(logits)
        pt = prob * labels + (1 - prob) * (1 - labels)
        alpha = pos_weight.to(logits.device) * labels + (1 - labels)
        loss = (alpha * (1 - pt).pow(GAMMA) * bce).mean()
        return (loss, output) if return_outputs else loss

args = TrainingArguments(
    output_dir=str(OUTPUT / 'checkpoints'), eval_strategy='epoch', save_strategy='epoch',
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model='micro_f1',
    greater_is_better=True, logging_steps=20, report_to='none', seed=SEED)
trainer = FocalTrainer(model=model, args=args, train_dataset=fit_ds,
                       eval_dataset=dev_ds, compute_metrics=metrics)
trainer.train()
print('내부 검증', trainer.evaluate())


## 5 봉인 평가 한 번

이 셀에서 처음으로 `test_sentence`에 예측한다. 기본 임계값 0.5를 고정한다. `조건` D·M은 채점에서 제외하고 제외 건수를 함께 출력한다. `근거_후보`는 예측과 가장 겹치는 후보를 택한다. 이 평가를 보고 설정을 바꾸면 다음 실험의 최종 점수는 별도 봉인셋이 필요하다.

In [ ]:
from collections import defaultdict

scored_rows = [r for r in test_rows if r.get('조건') not in ('D','M')]
held_rows = [r for r in test_rows if r.get('조건') in ('D','M')]
test_ds = make_dataset(scored_rows)
logits = trainer.predict(test_ds).predictions
prob = 1 / (1 + np.exp(-logits))
pred_matrix = prob > THRESHOLD

def selected_truth(r, pred):
    truth = {ho_key(c) for c in r['근거']}
    if not truth and r.get('근거_후보'):
        candidates = [{ho_key(c) for c in candidate} for candidate in r['근거_후보']]
        truth = max(candidates, key=lambda c: len(c & pred))
    return truth

gold = defaultdict(int); tp = defaultdict(int); fp = defaultdict(int)
clean_n = clean_fp = 0
for r, row_pred in zip(scored_rows, pred_matrix):
    pred = {all_ho[i] for i, hit in enumerate(row_pred) if hit}
    truth = selected_truth(r, pred)
    laws = {c.split(':',1)[0] for c in truth}
    for c in truth:
        gold[c] += 1
        tp[c] += int(c in pred)
    for c in pred - truth:
        if not truth or c.split(':',1)[0] in laws:
            fp[c] += 1
    if not r.get('조건') and not r['근거'] and not r.get('근거_후보'):
        clean_n += 1
        clean_fp += int(bool(pred))

print('평가 전체', len(test_rows), '채점', len(scored_rows),
      '제외', len(held_rows), dict(__import__('collections').Counter(r['조건'] for r in held_rows)))
print('적법 오탐', clean_fp, '/', clean_n, '=', clean_fp / clean_n if clean_n else 'N/A')
print(f'{"호":30s} {"정답":>5s} {"TP":>5s} {"FP":>5s} {"P":>7s} {"R":>7s} {"F1":>7s} 상태')
for c in sorted(set(gold) | set(fp)):
    g,t,f = gold[c],tp[c],fp[c]
    p = t/(t+f) if t+f else 0
    rec = t/g if g else 0
    f1 = 2*p*rec/(p+rec) if p+rec else 0
    status = '측정 가능' if g >= 30 else '측정 불가(<30)'
    print(f'{c:30s} {g:5d} {t:5d} {f:5d} {p:7.3f} {rec:7.3f} {f1:7.3f} {status}')


## 6 별도 저장

기존 09-22 최종 ZIP을 덮어쓰지 않는다. 출력 폴더에는 학습 설정, 라벨 순서와 두 입력 파일의 SHA-256을 함께 기록한다. 재학습 모델을 공식 최종본으로 채택할지는 평가와 팀 검토 뒤에 결정한다.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
save_dir = OUTPUT / 'model'
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
run_info = {
    'status': '2차 골든셋 재학습 실험 — 기존 09-22 모델 대체 아님',
    'base_model': MODEL_NAME, 'task': '조문 호 다중 라벨',
    'gamma': GAMMA, 'pos_weight_cap': POS_WEIGHT_CAP,
    'threshold': THRESHOLD, 'epochs': 3, 'learning_rate': 2e-5,
    'seed': SEED, 'labels': all_ho,
    'golden_sha256': sha256(GOLDEN), 'manifest_sha256': sha256(MANIFEST),
    'rows': {'all': len(rows), 'fit': len(fit_rows), 'dev': len(dev_rows),
             'test_sentence': len(test_rows), 'test_scored': len(scored_rows)},
    'note': '임계값은 봉인 평가셋에서 최적화하지 않음. 조건 D/M 제외. 30건 미만 호는 측정 불가.',
}
(save_dir / 'copylane_run.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장', save_dir.resolve())
print('모델 가중치와 copylane_run.json 확인:', sorted(p.name for p in save_dir.iterdir()))
